# CMPE 58T — Turkish Dependency Parsing
## 04 — Adapter-Based Multi-Treebank Training

**Architecture:** Houlsby adapters (bottleneck=64) injected into all 12 BERTurk layers.
Two adapters per layer (after attention, after FFN), one set per treebank (BOUN, IMST).
BERTurk weights frozen; only adapter parameters and the shared biaffine head are updated.

**Training:** Round-robin schedule — BOUN and IMST batches alternate. Per-treebank AdamW
optimizers with linear warmup. Best adapter weights selected by dev LAS.

**Key design decisions (consistent with 02_baseline and 03_ablation):**
- `run_inference_adapter` activates the adapter then uses `parser.predict()` — not the broken manual batch loop
- UAS/LAS from `parser.evaluate()` after activating the adapter — authoritative
- `force_rerun=True` in eval cell
- `parser.args.update(SUPAR_EXTRA)` before every evaluate call

---
## 0 — Mount Drive

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os, glob

PROJECT_DIR  = "/content/drive/MyDrive/58T - Application Project"
DATA_DIR     = os.path.join(PROJECT_DIR, "data")
MODEL_DIR    = os.path.join(PROJECT_DIR, "models")
ADAPTER_DIR  = os.path.join(MODEL_DIR,   "adapters")
PRED_DIR     = os.path.join(PROJECT_DIR, "predictions")
RESULTS_DIR  = os.path.join(PROJECT_DIR, "results")

for d in [DATA_DIR, MODEL_DIR, ADAPTER_DIR, PRED_DIR, RESULTS_DIR]:
    os.makedirs(d, exist_ok=True)

def get_splits(tb_name):
    folder = os.path.join(DATA_DIR, f"UD_Turkish-{tb_name}")
    files  = sorted(glob.glob(os.path.join(folder, "*.conllu")))
    splits = {}
    for f in files:
        name = os.path.basename(f)
        if   "train" in name: splits["train"] = f
        elif "dev"   in name: splits["dev"]   = f
        elif "test"  in name: splits["test"]  = f
    return splits

SPLITS = {tb: get_splits(tb) for tb in ["BOUN", "IMST", "PUD"]}

BASELINE_BOUN = os.path.join(MODEL_DIR,   "baseline_boun.pt")
BASE_MODEL    = os.path.join(ADAPTER_DIR, "adapter_base.pt")
ADAPTER_PATHS = {
    "BOUN": os.path.join(ADAPTER_DIR, "adapter_boun.pt"),
    "IMST": os.path.join(ADAPTER_DIR, "adapter_imst.pt"),
}

if not os.path.exists(BASELINE_BOUN):
    raise FileNotFoundError("baseline_boun.pt not found. Run 02_baseline.ipynb first.")
print(f"baseline_boun.pt : OK ({os.path.getsize(BASELINE_BOUN):,} B)")
print()
for tb, path in ADAPTER_PATHS.items():
    status = f"exists ({os.path.getsize(path):,} B)" if os.path.exists(path) else "new — will train"
    print(f"  adapter_{tb.lower()}.pt : {status}")
print(f"  adapter_base.pt  : {'exists' if os.path.exists(BASE_MODEL) else 'will be copied from baseline'}")


Mounted at /content/drive
baseline_boun.pt : OK (457,210,255 B)

  adapter_boun.pt : new — will train
  adapter_imst.pt : new — will train
  adapter_base.pt  : will be copied from baseline


---
## 1 — Install & Patch

In [2]:
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118 -q
!pip install "transformers==4.35.2" -q
!pip install supar -q
!pip install conllu -q
!pip install scipy -q

import os
PARSER_PY = "/usr/local/lib/python3.12/dist-packages/supar/parsers/parser.py"
OLD = 'state = torch.load(path if os.path.exists(path) else download(supar.MODEL[src].get(path, path), reload=reload))'
NEW = 'state = torch.load(path if os.path.exists(path) else download(supar.MODEL[src].get(path, path), reload=reload), weights_only=False)'
with open(PARSER_PY) as f: src = f.read()
if OLD in src:
    with open(PARSER_PY, 'w') as f: f.write(src.replace(OLD, NEW))
    print("Patch applied")
else:
    print("Patch already applied")

from transformers import AdamW, get_linear_schedule_with_warmup
import torch, transformers
print(f"transformers : {transformers.__version__}")
print(f"PyTorch      : {torch.__version__}")
print(f"GPU          : {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE — switch runtime!'}")


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 123.5/123.5 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.9/7.9 MB 84.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 44.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 109.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
sentence-transformers 5.4.1 requires transformers<6.0.0,>=4.41.0, but you have transformers 4.35.2 which is incompatible.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 93.1/93.1 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 37.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 773.7/773.7 kB 61.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 608.4/608.4 kB 47.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 418.7/418.7 kB 34.6 MB/s 

/usr/local/lib/python3.12/dist-packages/transformers/utils/generic.py:441: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(


transformers : 4.35.2
PyTorch      : 2.10.0+cu128
GPU          : NVIDIA A100-SXM4-40GB


---
## 2 — Hyperparameters

In [3]:
import torch

BERTURK_NAME = "dbmdz/bert-base-turkish-uncased"
TREEBANKS    = ["BOUN", "IMST"]   # PUD is test-only
BOTTLENECK   = 64                  # adapter bottleneck dim (UDapter uses 64)
DEVICE       = "cuda" if torch.cuda.is_available() else "cpu"

BATCH_SIZE   = 5000
MAX_EPOCHS   = 30
PATIENCE     = 10
LR_ADAPTER   = 1e-3    # adapter + biaffine head LR
WARMUP       = 0.1
WEIGHT_DECAY = 0.01
EPS          = 1e-8
GRAD_CLIP    = 5.0

SUPAR_EXTRA = dict(
    mbr=True, partial=False, tree=False,
    proj=False, punct=False, update_steps=1, clip=5.0
)

print(f"BOTTLENECK={BOTTLENECK}  LR_ADAPTER={LR_ADAPTER}  MAX_EPOCHS={MAX_EPOCHS}  PATIENCE={PATIENCE}")
print(f"DEVICE={DEVICE}  BATCH={BATCH_SIZE}")


BOTTLENECK=64  LR_ADAPTER=0.001  MAX_EPOCHS=30  PATIENCE=10
DEVICE=cuda  BATCH=5000


---
## 3 — Adapter Classes

In [4]:
import torch
import torch.nn as nn

class HoulsbyAdapter(nn.Module):
    """Single Houlsby adapter: LayerNorm -> down -> GELU -> up + residual."""
    def __init__(self, hidden_size, bottleneck_dim=64):
        super().__init__()
        self.layer_norm = nn.LayerNorm(hidden_size)
        self.down_proj  = nn.Linear(hidden_size, bottleneck_dim)
        self.activation = nn.GELU()
        self.up_proj    = nn.Linear(bottleneck_dim, hidden_size)
        nn.init.normal_(self.down_proj.weight, std=1e-3)
        nn.init.zeros_(self.down_proj.bias)
        nn.init.normal_(self.up_proj.weight, std=1e-3)
        nn.init.zeros_(self.up_proj.bias)

    def forward(self, x):
        residual = x
        x = self.layer_norm(x)
        x = self.up_proj(self.activation(self.down_proj(x)))
        return x + residual


class BertLayerWithAdapters(nn.Module):
    """Wraps BertLayer with two Houlsby adapters (after attention and after FFN)."""
    def __init__(self, bert_layer, hidden_size, bottleneck_dim=64):
        super().__init__()
        self.bert_layer = bert_layer
        self.adapter_a  = HoulsbyAdapter(hidden_size, bottleneck_dim)
        self.adapter_b  = HoulsbyAdapter(hidden_size, bottleneck_dim)

    def forward(self, hidden_states, attention_mask=None, *args, **kwargs):
        layer = self.bert_layer
        self_attn_outputs = layer.attention(hidden_states, attention_mask, *args, **kwargs)
        attn_output = self.adapter_a(self_attn_outputs[0])
        intermediate_output = layer.intermediate(attn_output)
        layer_output = self.adapter_b(layer.output(intermediate_output, attn_output))
        return (layer_output,) + self_attn_outputs[1:]


def inject_adapters(bert_model, treebanks, bottleneck_dim, device):
    """Freeze BERTurk and inject per-treebank adapter modules into all layers."""
    hidden_size = bert_model.config.hidden_size
    for param in bert_model.parameters():
        param.requires_grad = False

    adapter_store = {}
    for i in range(bert_model.config.num_hidden_layers):
        original_layer = bert_model.encoder.layer[i]
        wrapper = BertLayerWithAdapters(original_layer, hidden_size, bottleneck_dim)
        adapter_store[i] = {
            tb: {"a": HoulsbyAdapter(hidden_size, bottleneck_dim).to(device),
                 "b": HoulsbyAdapter(hidden_size, bottleneck_dim).to(device)}
            for tb in treebanks
        }
        wrapper.adapter_a = adapter_store[i][treebanks[0]]["a"]
        wrapper.adapter_b = adapter_store[i][treebanks[0]]["b"]
        bert_model.encoder.layer[i] = wrapper

    return adapter_store


def activate_adapter(bert_model, adapter_store, treebank):
    """Switch active adapter weights to the given treebank."""
    for i, layer in enumerate(bert_model.encoder.layer):
        if isinstance(layer, BertLayerWithAdapters):
            layer.adapter_a = adapter_store[i][treebank]["a"]
            layer.adapter_b = adapter_store[i][treebank]["b"]


def save_adapters(adapter_store, treebank, path):
    state = {}
    for i in adapter_store:
        state[f"layer_{i}_a"] = adapter_store[i][treebank]["a"].state_dict()
        state[f"layer_{i}_b"] = adapter_store[i][treebank]["b"].state_dict()
    torch.save(state, path)


def load_adapters(adapter_store, treebank, path, device):
    state = torch.load(path, map_location=device, weights_only=False)
    for i in adapter_store:
        adapter_store[i][treebank]["a"].load_state_dict(state[f"layer_{i}_a"])
        adapter_store[i][treebank]["b"].load_state_dict(state[f"layer_{i}_b"])


def get_adapter_params(adapter_store, treebank):
    params = []
    for i in adapter_store:
        params += list(adapter_store[i][treebank]["a"].parameters())
        params += list(adapter_store[i][treebank]["b"].parameters())
    return params


print("Adapter classes and helpers defined.")


Adapter classes and helpers defined.


---
## 4 — Inference & Scoring Helpers

`run_inference_adapter`: activates adapter → `parser.predict()` → reads file back.
`evaluate_and_save_adapter`: activates adapter → `parser.evaluate()` for UAS/LAS → `score_predictions()` for npy only.

In [5]:
import numpy as np
import conllu as conllu_lib
import json
from collections import defaultdict
from supar import CRF2oDependencyParser
from supar.utils import Dataset

def load_gold(conllu_path):
    with open(conllu_path, encoding="utf-8") as f:
        return conllu_lib.parse(f.read())

def make_loader(parser, conllu_path, batch_size=5000, shuffle=False):
    dataset = Dataset(parser.transform, conllu_path)
    dataset.build(batch_size, 32)
    return dataset.loader

def run_inference_adapter(parser, adapter_store, adapter_tb,
                           conllu_path, pred_conllu_path, force_rerun=False):
    """
    Activate the given adapter, then use parser.predict() to write predictions.
    Returns parsed predicted sentences.
    Uses cached file if force_rerun=False and file already exists.
    """
    if (not force_rerun
            and os.path.exists(pred_conllu_path)
            and os.path.getsize(pred_conllu_path) > 0):
        print(f"    Cached: {os.path.basename(pred_conllu_path)}")
        with open(pred_conllu_path, encoding="utf-8") as f:
            return conllu_lib.parse(f.read())

    activate_adapter(parser.model.encoder.bert, adapter_store, adapter_tb)
    parser.predict(conllu_path, pred=pred_conllu_path, verbose=False)

    with open(pred_conllu_path, encoding="utf-8") as f:
        pred_sents = conllu_lib.parse(f.read())

    print(f"    Saved: {os.path.basename(pred_conllu_path)} ({len(pred_sents)} sentences)")
    return pred_sents

def score_predictions(gold_sents, pred_sents):
    """Sentence-level boolean arrays for McNemar's tests only."""
    total = correct_arc = correct_both = 0
    sent_uas, sent_las = [], []
    for gold, pred in zip(gold_sents, pred_sents):
        gt = [t for t in gold if isinstance(t["id"], int)]
        pt = [t for t in pred if isinstance(t["id"], int)]
        n = len(gt)
        if n == 0: continue
        arc_ok  = sum(1 for g, p in zip(gt, pt) if g["head"] == p["head"])
        both_ok = sum(1 for g, p in zip(gt, pt)
                      if g["head"] == p["head"] and g["deprel"] == p["deprel"])
        total += n; correct_arc += arc_ok; correct_both += both_ok
        sent_uas.append(arc_ok == n); sent_las.append(both_ok == n)
    uas = round(100 * correct_arc  / total, 2) if total else 0.0
    las = round(100 * correct_both / total, 2) if total else 0.0
    return uas, las, np.array(sent_uas, dtype=bool), np.array(sent_las, dtype=bool)

def per_relation_las(gold_sents, pred_sents):
    counts = defaultdict(lambda: [0, 0])
    for gold, pred in zip(gold_sents, pred_sents):
        gt = [t for t in gold if isinstance(t["id"], int)]
        pt = [t for t in pred if isinstance(t["id"], int)]
        for g, p in zip(gt, pt):
            counts[g["deprel"]][1] += 1
            if g["head"] == p["head"] and g["deprel"] == p["deprel"]:
                counts[g["deprel"]][0] += 1
    return {rel: {"correct": c, "total": t,
                  "LAS": round(100 * c / t, 2) if t else 0.0}
            for rel, (c, t) in sorted(counts.items(), key=lambda x: -x[1][1])}

def evaluate_and_save_adapter(parser, adapter_store, adapter_tb,
                               system_name, conllu_path, test_name,
                               force_rerun=False):
    """
    Write predictions with parser.predict() (adapter active).
    Score with parser.evaluate() for authoritative UAS/LAS.
    Save CoNLL-U and per-sentence npy for McNemar's tests.
    """
    safe        = system_name.replace(" ", "_")
    pred_conllu = os.path.join(PRED_DIR, f"{safe}_{test_name}.conllu")
    pred_npy    = os.path.join(PRED_DIR, f"{safe}_{test_name}_sent_las.npy")

    print(f"  [{system_name}] → {test_name}  (force_rerun={force_rerun})")
    pred_sents = run_inference_adapter(
        parser, adapter_store, adapter_tb,
        conllu_path, pred_conllu, force_rerun=force_rerun)

    # Authoritative UAS/LAS — activate adapter then use parser.evaluate()
    activate_adapter(parser.model.encoder.bert, adapter_store, adapter_tb)
    parser.args.update(SUPAR_EXTRA)
    _, metric = parser.evaluate(conllu_path, verbose=False)
    uas = round(metric.uas * 100, 2)
    las = round(metric.las * 100, 2)

    # Sentence-level arrays for McNemar's — load gold fresh
    gold_fresh = load_gold(conllu_path)
    _, _, s_uas, s_las = score_predictions(gold_fresh, pred_sents)
    np.save(pred_npy, s_las)

    print(f"    UAS {uas:.2f}  LAS {las:.2f}  npy: {os.path.basename(pred_npy)}")
    return {"system": system_name, "test": test_name, "UAS": uas, "LAS": las,
            "pred_conllu": os.path.basename(pred_conllu),
            "pred_npy":    os.path.basename(pred_npy)}

# Load gold sentences once
gold = {
    "BOUN": load_gold(SPLITS["BOUN"]["test"]),
    "IMST": load_gold(SPLITS["IMST"]["test"]),
    "PUD" : load_gold(SPLITS["PUD"]["test"]),
}
print("Helpers defined.")
for tb, sents in gold.items():
    print(f"  Gold {tb}: {len(sents)} sentences")


Helpers defined.
  Gold BOUN: 979 sentences
  Gold IMST: 1100 sentences
  Gold PUD: 1000 sentences


---
## 5 — Load Base Model

Copies `baseline_boun.pt` as the shared base. BERTurk weights will be frozen during adapter injection.

In [6]:
from supar import CRF2oDependencyParser
import shutil

if os.path.exists(BASE_MODEL):
    print(f"Loading existing base model: {BASE_MODEL}")
    parser = CRF2oDependencyParser.load(BASE_MODEL)
else:
    print("Copying baseline_boun.pt -> adapter_base.pt")
    shutil.copy2(BASELINE_BOUN, BASE_MODEL)
    parser = CRF2oDependencyParser.load(BASE_MODEL)
    print("Base model saved.")

parser.args.update(SUPAR_EXTRA)
parser.model = parser.model.to(DEVICE)
print(f"Base parser loaded on {DEVICE}.")
print(f"  Encoder : {parser.args.bert}")


Copying baseline_boun.pt -> adapter_base.pt


/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/59.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

/usr/local/lib/python3.12/dist-packages/transformers/utils/generic.py:309: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
/usr/local/lib/python3.12/dist-packages/transformers/utils/generic.py:309: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


pytorch_model.bin:   0%|          | 0.00/445M [00:00<?, ?B/s]

Base model saved.
Base parser loaded on cuda.
  Encoder : dbmdz/bert-base-turkish-uncased


---
## 6 — Inject Adapters

In [7]:
bert_model    = parser.model.encoder.bert
adapter_store = inject_adapters(bert_model, TREEBANKS, BOTTLENECK, DEVICE)

total_params    = sum(p.numel() for p in parser.model.parameters())
frozen_params   = sum(p.numel() for p in parser.model.parameters() if not p.requires_grad)
adapter_params  = sum(
    sum(p.numel() for p in adapter_store[i][tb][k].parameters())
    for i in adapter_store for tb in TREEBANKS for k in ["a", "b"])
biaffine_params = sum(p.numel() for p in parser.model.parameters() if p.requires_grad)

print(f"BERTurk injected ({bert_model.config.num_hidden_layers} layers x 2 adapters x {len(TREEBANKS)} treebanks)")
print(f"  Bottleneck dim      : {BOTTLENECK}")
print(f"  Total model params  : {total_params:,}")
print(f"  Frozen (BERTurk)    : {frozen_params:,}")
print(f"  Adapter params each : {adapter_params//len(TREEBANKS):,}  (x{len(TREEBANKS)} = {adapter_params:,} total)")
print(f"  Biaffine head       : {biaffine_params:,}")


BERTurk injected (12 layers x 2 adapters x 2 treebanks)
  Bottleneck dim      : 64
  Total model params  : 116,537,049
  Frozen (BERTurk)    : 110,617,344
  Adapter params each : 2,416,128  (x2 = 4,832,256 total)
  Biaffine head       : 5,919,705


---
## 7 — Build Data Loaders

In [8]:
print("Building data loaders...")
train_loaders = {}
dev_loaders   = {}

for tb in TREEBANKS:
    print(f"  Loading {tb}...")
    train_loaders[tb] = make_loader(parser, SPLITS[tb]["train"], BATCH_SIZE, shuffle=True)
    dev_loaders[tb]   = make_loader(parser, SPLITS[tb]["dev"],   BATCH_SIZE, shuffle=False)
    print(f"    train: {len(train_loaders[tb])} batches  dev: {len(dev_loaders[tb])} batches")


Building data loaders...
  Loading BOUN...


/usr/local/lib/python3.12/dist-packages/supar/utils/fn.py:155: UserWarning: Using a non-tuple sequence for multidimensional indexing is deprecated and will be changed in pytorch 2.9; use x[tuple(seq)] instead of x[seq]. In pytorch 2.9 this will be interpreted as tensor index, x[torch.tensor(seq)], which will result either in an error or a different result (Triggered internally at /pytorch/torch/csrc/autograd/python_variable_indexing.cpp:347.)
  out_tensor[i][[slice(-i, None) if padding_side == 'left' else slice(0, i) for i in tensor.size()]] = tensor


    train: 43 batches  dev: 32 batches
  Loading IMST...
    train: 32 batches  dev: 32 batches


---
## 8 — Build Optimizers

In [9]:
from transformers import AdamW, get_linear_schedule_with_warmup

biaffine_params = [p for p in parser.model.parameters() if p.requires_grad]
steps_per_epoch = max(len(train_loaders[tb]) for tb in TREEBANKS)
total_steps     = steps_per_epoch * MAX_EPOCHS
warmup_steps    = int(total_steps * WARMUP)

optimizers = {}
schedulers = {}
for tb in TREEBANKS:
    opt = AdamW(
        get_adapter_params(adapter_store, tb) + biaffine_params,
        lr=LR_ADAPTER, weight_decay=WEIGHT_DECAY, eps=EPS)
    sch = get_linear_schedule_with_warmup(opt, warmup_steps, total_steps)
    optimizers[tb] = opt
    schedulers[tb] = sch

print(f"Optimizers built for: {TREEBANKS}")
print(f"  Total steps    : {total_steps}")
print(f"  Warmup steps   : {warmup_steps}")
print(f"  Steps/epoch    : {steps_per_epoch}")


/usr/local/lib/python3.12/dist-packages/transformers/optimization.py:411: FutureWarning: This implementation of AdamW is deprecated and will be removed in a future version. Use the PyTorch implementation torch.optim.AdamW instead, or set `no_deprecation_warning=True` to disable this warning
  warnings.warn(


Optimizers built for: ['BOUN', 'IMST']
  Total steps    : 1290
  Warmup steps   : 129
  Steps/epoch    : 43


/usr/local/lib/python3.12/dist-packages/torch/_compile.py:54: UserWarning: optimizer contains a parameter group with duplicate parameters; in future, this will cause an error; see github.com/pytorch/pytorch/issues/40967 for more information
  return disable_fn(*args, **kwargs)
/usr/local/lib/python3.12/dist-packages/transformers/optimization.py:411: FutureWarning: This implementation of AdamW is deprecated and will be removed in a future version. Use the PyTorch implementation torch.optim.AdamW instead, or set `no_deprecation_warning=True` to disable this warning
  warnings.warn(


---
## 9 — Dev Evaluation Helper

In [10]:
from supar.utils.metric import AttachmentMetric

def evaluate_dev(parser, loader, adapter_store, treebank):
    """Evaluate on dev set with given adapter active. Returns (loss, UAS, LAS)."""
    activate_adapter(parser.model.encoder.bert, adapter_store, treebank)
    parser.model.eval()
    total_loss = 0
    metric     = AttachmentMetric()

    with torch.no_grad():
        for batch in loader:
            words, texts, *feats, arcs, sibs, rels = batch
            word_mask = words.ne(parser.args.pad_index)
            mask = word_mask if len(words.shape) < 3 else word_mask.any(-1)
            mask[:, 0] = 0
            words = words.to(DEVICE); feats = [f.to(DEVICE) for f in feats]
            arcs  = arcs.to(DEVICE);  sibs  = sibs.to(DEVICE)
            rels  = rels.to(DEVICE);  mask  = mask.to(DEVICE)
            s_arc, s_sib, s_rel = parser.model(words, feats)
            loss, s_arc = parser.model.loss(
                s_arc, s_sib, s_rel, arcs, sibs, rels, mask,
                parser.args.mbr, parser.args.partial)
            arc_preds, rel_preds = parser.model.decode(
                s_arc, s_sib, s_rel, mask,
                parser.args.tree, parser.args.mbr, parser.args.proj)
            total_loss += loss.item()
            metric(arc_preds, rel_preds, arcs, rels, mask)

    return total_loss / len(loader), metric.uas * 100, metric.las * 100

print("evaluate_dev() defined.")


evaluate_dev() defined.


---
## 10 — Train (Round-Robin)

Skips training if both adapter checkpoints already exist on Drive.

In [11]:
import json

adapters_exist = all(os.path.exists(ADAPTER_PATHS[tb]) for tb in TREEBANKS)

if adapters_exist:
    print("All adapter checkpoints found — skipping training.")
    for tb in TREEBANKS:
        load_adapters(adapter_store, tb, ADAPTER_PATHS[tb], DEVICE)
        print(f"  Loaded {tb} adapter")
    history = []
else:
    best_las    = {tb: 0.0 for tb in TREEBANKS}
    patience_ct = {tb: 0   for tb in TREEBANKS}
    history     = []

    print(f"Round-robin training: {MAX_EPOCHS} epochs, patience {PATIENCE}")
    print("=" * 70)

    for epoch in range(1, MAX_EPOCHS + 1):
        print(f"\nEpoch {epoch} / {MAX_EPOCHS}")
        parser.model.train()
        epoch_loss = {tb: 0.0 for tb in TREEBANKS}
        epoch_n    = {tb: 0   for tb in TREEBANKS}

        iters = {tb: iter(train_loaders[tb]) for tb in TREEBANKS}
        for batches in zip(*[iters[tb] for tb in TREEBANKS]):
            for tb, batch in zip(TREEBANKS, batches):
                activate_adapter(parser.model.encoder.bert, adapter_store, tb)
                words, texts, *feats, arcs, sibs, rels = batch
                word_mask = words.ne(parser.args.pad_index)
                mask = word_mask if len(words.shape) < 3 else word_mask.any(-1)
                mask[:, 0] = 0
                words = words.to(DEVICE); feats = [f.to(DEVICE) for f in feats]
                arcs  = arcs.to(DEVICE);  sibs  = sibs.to(DEVICE)
                rels  = rels.to(DEVICE);  mask  = mask.to(DEVICE)

                s_arc, s_sib, s_rel = parser.model(words, feats)
                loss, _ = parser.model.loss(
                    s_arc, s_sib, s_rel, arcs, sibs, rels, mask,
                    parser.args.mbr, parser.args.partial)

                optimizers[tb].zero_grad()
                loss.backward()
                nn.utils.clip_grad_norm_(
                    get_adapter_params(adapter_store, tb) +
                    [p for p in parser.model.parameters() if p.requires_grad],
                    GRAD_CLIP)
                optimizers[tb].step()
                schedulers[tb].step()
                epoch_loss[tb] += loss.item()
                epoch_n[tb]    += 1

        epoch_record = {"epoch": epoch}
        for tb in TREEBANKS:
            avg_loss = epoch_loss[tb] / max(epoch_n[tb], 1)
            dev_loss, uas, las = evaluate_dev(parser, dev_loaders[tb], adapter_store, tb)
            tag = ""
            if las > best_las[tb]:
                best_las[tb]    = las
                patience_ct[tb] = 0
                save_adapters(adapter_store, tb, ADAPTER_PATHS[tb])
                tag = " (saved)"
            else:
                patience_ct[tb] += 1
                tag = f" (patience {patience_ct[tb]}/{PATIENCE})"
            print(f"  {tb:<6} train {avg_loss:.4f} | dev {dev_loss:.4f} | UAS {uas:.2f} LAS {las:.2f}{tag}")
            epoch_record[tb] = {"train_loss": avg_loss, "dev_uas": uas, "dev_las": las}

        history.append(epoch_record)

        if all(patience_ct[tb] >= PATIENCE for tb in TREEBANKS):
            print(f"\nEarly stopping at epoch {epoch}.")
            break

    print("\nTraining complete.")
    print("Best dev LAS: " + "  ".join(f"{tb}: {best_las[tb]:.2f}" for tb in TREEBANKS))

hist_path = os.path.join(RESULTS_DIR, "adapter_training_history.json")
with open(hist_path, "w") as f:
    json.dump(history, f, indent=2)
print(f"History saved: {hist_path}")


/usr/local/lib/python3.12/dist-packages/supar/utils/fn.py:155: UserWarning: Using a non-tuple sequence for multidimensional indexing is deprecated and will be changed in pytorch 2.9; use x[tuple(seq)] instead of x[seq]. In pytorch 2.9 this will be interpreted as tensor index, x[torch.tensor(seq)], which will result either in an error or a different result (Triggered internally at /pytorch/torch/csrc/autograd/python_variable_indexing.cpp:347.)
  out_tensor[i][[slice(-i, None) if padding_side == 'left' else slice(0, i) for i in tensor.size()]] = tensor


Round-robin training: 30 epochs, patience 10

Epoch 1 / 30
  BOUN   train 93.1428 | dev 16.0231 | UAS 53.74 LAS 22.47 (saved)
  IMST   train 184.7808 | dev 24.8986 | UAS 65.30 LAS 17.25 (saved)

Epoch 2 / 30
  BOUN   train 4.2109 | dev 2.0477 | UAS 76.30 LAS 60.98 (saved)
  IMST   train 15.4134 | dev 1.9432 | UAS 67.91 LAS 50.26 (saved)

Epoch 3 / 30
  BOUN   train 1.2279 | dev 1.8063 | UAS 78.53 LAS 67.57 (saved)
  IMST   train 2.0664 | dev 1.5353 | UAS 74.85 LAS 61.08 (saved)

Epoch 4 / 30
  BOUN   train 0.9093 | dev 1.8194 | UAS 80.13 LAS 70.66 (saved)
  IMST   train 1.6335 | dev 1.3146 | UAS 75.58 LAS 64.50 (saved)

Epoch 5 / 30
  BOUN   train 0.7754 | dev 1.7973 | UAS 79.94 LAS 71.19 (saved)
  IMST   train 1.3343 | dev 1.2499 | UAS 76.83 LAS 65.19 (saved)

Epoch 6 / 30
  BOUN   train 0.7068 | dev 1.6817 | UAS 80.89 LAS 72.58 (saved)
  IMST   train 1.1465 | dev 1.3319 | UAS 78.49 LAS 67.27 (saved)

Epoch 7 / 30
  BOUN   train 0.6253 | dev 1.5542 | UAS 80.75 LAS 72.73 (saved)
  IMST

---
## 11 — Evaluate: Inference + Predictions

`force_rerun=True` ensures fresh predictions from the best saved adapters.

In [12]:
import json

# Load best adapter weights
for tb in TREEBANKS:
    if not os.path.exists(ADAPTER_PATHS[tb]):
        raise FileNotFoundError(f"{ADAPTER_PATHS[tb]} not found — run section 10 first.")
    load_adapters(adapter_store, tb, ADAPTER_PATHS[tb], DEVICE)
    print(f"Loaded best {tb} adapter")

results = []

# BOUN adapter: BOUN test + PUD zero-shot
for test_name, test_path in [("BOUN", SPLITS["BOUN"]["test"]),
                              ("PUD",  SPLITS["PUD"]["test"])]:
    r = evaluate_and_save_adapter(
        parser, adapter_store, "BOUN",
        "adapter_boun", test_path, test_name, force_rerun=True)
    results.append(r)

# IMST adapter: IMST test + PUD zero-shot
for test_name, test_path in [("IMST", SPLITS["IMST"]["test"]),
                              ("PUD",  SPLITS["PUD"]["test"])]:
    r = evaluate_and_save_adapter(
        parser, adapter_store, "IMST",
        "adapter_imst", test_path, test_name, force_rerun=True)
    results.append(r)

results_path = os.path.join(RESULTS_DIR, "adapter_results.json")
with open(results_path, "w") as f:
    json.dump(results, f, indent=2)
print(f"\nResults saved: {results_path}")


Loaded best BOUN adapter
Loaded best IMST adapter
  [adapter_boun] → BOUN  (force_rerun=True)
    Saved: adapter_boun_BOUN.conllu (979 sentences)
    UAS 77.96  LAS 68.74  npy: adapter_boun_BOUN_sent_las.npy
  [adapter_boun] → PUD  (force_rerun=True)
    Saved: adapter_boun_PUD.conllu (1000 sentences)
    UAS 76.23  LAS 63.87  npy: adapter_boun_PUD_sent_las.npy
  [adapter_imst] → IMST  (force_rerun=True)
    Saved: adapter_imst_IMST.conllu (1100 sentences)
    UAS 80.65  LAS 70.08  npy: adapter_imst_IMST_sent_las.npy
  [adapter_imst] → PUD  (force_rerun=True)
    Saved: adapter_imst_PUD.conllu (1000 sentences)
    UAS 74.09  LAS 62.72  npy: adapter_imst_PUD_sent_las.npy

Results saved: /content/drive/MyDrive/58T - Application Project/results/adapter_results.json


---
## 12 — Results Table

In [13]:
print(f"{'System':<20} {'Test':<6} {'UAS':>7} {'LAS':>7}  Predictions")
print("-" * 75)
for r in results:
    print(f"  {r['system']:<18} {r['test']:<6} {r['UAS']:>7.2f} {r['LAS']:>7.2f}  {r['pred_conllu']}")


System               Test       UAS     LAS  Predictions
---------------------------------------------------------------------------
  adapter_boun       BOUN     77.96   68.74  adapter_boun_BOUN.conllu
  adapter_boun       PUD      76.23   63.87  adapter_boun_PUD.conllu
  adapter_imst       IMST     80.65   70.08  adapter_imst_IMST.conllu
  adapter_imst       PUD      74.09   62.72  adapter_imst_PUD.conllu


---
## 13 — Per-Relation LAS Breakdown (BOUN adapter on BOUN test)

In [14]:
import json

pred_path = os.path.join(PRED_DIR, "adapter_boun_BOUN.conllu")
if not os.path.exists(pred_path):
    print("Prediction file not found — run section 11 first.")
else:
    gold_boun = load_gold(SPLITS["BOUN"]["test"])
    with open(pred_path, encoding="utf-8") as f:
        pred_sents = conllu_lib.parse(f.read())
    breakdown = per_relation_las(gold_boun, pred_sents)
    br_path = os.path.join(RESULTS_DIR, "adapter_boun_BOUN_per_rel.json")
    with open(br_path, "w") as f:
        json.dump(breakdown, f, indent=2)
    print(f"Per-relation saved: {br_path}")
    print()
    print(f"{'Relation':<15} {'Count':>7} {'LAS':>7}")
    print("-" * 32)
    for rel, vals in list(breakdown.items())[:20]:
        print(f"  {rel:<13} {vals['total']:>7} {vals['LAS']:>7.2f}")


Per-relation saved: /content/drive/MyDrive/58T - Application Project/results/adapter_boun_BOUN_per_rel.json

Relation          Count     LAS
--------------------------------
  punct            2026   74.78
  nmod:poss        1053   82.43
  obl               987   65.96
  root              979   83.25
  nsubj             829   72.86
  amod              766   60.97
  obj               728   71.98
  conj              679   55.96
  det               524   89.89
  advmod            436   65.83
  nmod              366   34.15
  acl               351   66.67
  advcl             317   63.09
  case              294   69.05
  cc                258   64.73
  compound          217   42.40
  cop               192   77.60
  flat              181   74.03
  ccomp             173   71.68
  nummod            163   71.78


---
## 14 — Verify Saved Files

In [15]:
expected = [
    (ADAPTER_DIR,  "adapter_base.pt"),
    (ADAPTER_DIR,  "adapter_boun.pt"),
    (ADAPTER_DIR,  "adapter_imst.pt"),
    (PRED_DIR,     "adapter_boun_BOUN.conllu"),
    (PRED_DIR,     "adapter_boun_BOUN_sent_las.npy"),
    (PRED_DIR,     "adapter_boun_PUD.conllu"),
    (PRED_DIR,     "adapter_boun_PUD_sent_las.npy"),
    (PRED_DIR,     "adapter_imst_IMST.conllu"),
    (PRED_DIR,     "adapter_imst_IMST_sent_las.npy"),
    (PRED_DIR,     "adapter_imst_PUD.conllu"),
    (PRED_DIR,     "adapter_imst_PUD_sent_las.npy"),
    (RESULTS_DIR,  "adapter_results.json"),
    (RESULTS_DIR,  "adapter_boun_BOUN_per_rel.json"),
    (RESULTS_DIR,  "adapter_training_history.json"),
]

print(f"{'Status':<8} {'Size':>12}  {'Dir':<14} File")
print("-" * 85)
all_ok = True
for directory, filename in expected:
    path   = os.path.join(directory, filename)
    exists = os.path.exists(path)
    size   = f"{os.path.getsize(path):,} B" if exists else "-"
    status = "OK" if exists else "MISSING"
    folder = os.path.basename(directory)
    if not exists: all_ok = False
    print(f"  {status:<6} {size:>12}  {folder:<14} {filename}")

print()
if all_ok:
    print("All files present. 04_adapters complete.")
    print("Next: 05_morph_ud_feats.ipynb")
else:
    print("Some files missing — rerun the relevant sections above.")


Status           Size  Dir            File
-------------------------------------------------------------------------------------
  OK     457,210,255 B  adapters       adapter_base.pt
  OK      9,713,869 B  adapters       adapter_boun.pt
  OK      9,713,869 B  adapters       adapter_imst.pt
  OK        966,201 B  predictions    adapter_boun_BOUN.conllu
  OK          1,107 B  predictions    adapter_boun_BOUN_sent_las.npy
  OK      1,385,590 B  predictions    adapter_boun_PUD.conllu
  OK          1,128 B  predictions    adapter_boun_PUD_sent_las.npy
  OK        845,889 B  predictions    adapter_imst_IMST.conllu
  OK          1,228 B  predictions    adapter_imst_IMST_sent_las.npy
  OK      1,383,680 B  predictions    adapter_imst_PUD.conllu
  OK          1,128 B  predictions    adapter_imst_PUD_sent_las.npy
  OK            760 B  results        adapter_results.json
  OK          3,173 B  results        adapter_boun_BOUN_per_rel.json
  OK          8,668 B  results        adapter_training_h